[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/03_setup/03_setup_check.ipynb)


# 03 — 분석 환경 점검 (도구를 실제로 돌려 확인)

> 본문: [`03_setup.md`](03_setup.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 3초** (실측)

**이 노트북은 도구를 직접 돌립니다.** import 여부만 보는 게 아니라 **ANARCI 를 실제로 한 번 돌려** numbering 이 나오는지 확인해요.
내 결과는 `my_run/` 에 쌓이고 커밋된 `data/` 는 대조군이에요 — 어느 단계가 실패해도 `resolve()` 가 `data/` 로 폴백해 뒤 절이 계속 돌아갑니다.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

Colab 은 이 셀 하나로 끝나고, 로컬은 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "03_setup"
PIP_PKGS = "pandas anarci abnumber"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True    # ANARCI 계열은 hmmscan(HMMER) 실행파일이 필요해요 (pip 로는 안 깔려요)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후엔 cwd 아래만 깊이 3까지 — 위로 올라가 rglob 하면 Colab 에서 / 전체를 뒤집니다.
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False
_APT = []                                # 필요한 시스템 패키지를 모아 apt 를 한 번만 돌립니다

if shutil.which("hmmscan") is None:      # ANARCI 가 부르는 실행파일 — pip 로는 안 깔려요
    if IN_COLAB:
        _APT.append("hmmer")
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if _APT:
    _run("apt-get -qq update")           # 인덱스가 낡으면 install 이 404 로 죽습니다
    _run("apt-get -qq install -y " + " ".join(_APT))

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — 스택 점검 + ANARCI 스모크 테스트 (본문 3.5)

"설치됐다"의 진짜 기준은 **도구가 결과를 내놓는가**예요. 아래 셀은 이 챕터가 실제로 설치하는 것만
확인하고(부트스트랩의 `PIP_PKGS` 를 그대로 읽어요 — 점검 목록이 설치 목록과 어긋날 일이 없어요),
곧바로 데모 항체를 **실제로 numbering** 해서 걸린 시간까지 잽니다.

In [ ]:
import shutil, sys, time
import pandas as pd

rows, SMOKE = [], {}

def check(kind, name, ok, detail=""):
    rows.append({"kind": kind, "item": name, "ok": "O" if ok else "X", "detail": detail})
    print(("O " if ok else "X "), f"{name:12s}", detail)

print("[현재 커널 python]", sys.executable)
for pkg in PIP_PKGS.split():                       # 이 챕터가 pip 로 까는 것 그대로
    check("python", pkg, _have(pkg))
for tool in (["hmmscan", "ANARCI"] if NEED_HMMER else ["ANARCI"]):
    check("cli", tool, shutil.which(tool) is not None, shutil.which(tool) or "PATH에 없음")

# --- 스모크 테스트: 데모 항체를 실제로 numbering ---
try:
    from abnumber import Chain
    seqs, name = {}, None
    for line in open("data/demo_mab.fa"):
        line = line.strip()
        if line.startswith(">"): name = line[1:].split()[0]; seqs[name] = ""
        elif name: seqs[name] += line
    for sid, seq in seqs.items():
        t0 = time.time()
        ch = Chain(seq, scheme="imgt")
        dt = time.time() - t0
        SMOKE[sid] = {"chain_type": ch.chain_type, "cdr1_imgt": ch.cdr1_seq,
                      "cdr2_imgt": ch.cdr2_seq, "cdr3_imgt": ch.cdr3_seq,
                      "length": len(seq), "sec": round(dt, 3)}
        rows.append({"kind": "smoke", "item": sid, "ok": "O",
                     "detail": f"chain_type={ch.chain_type} cdr3={ch.cdr3_seq} ({dt:.2f}s)"})
        print(f"O  smoke        {sid} → chain_type {ch.chain_type} | CDR3 {ch.cdr3_seq} | {dt:.2f}s")
except Exception as e:
    rows.append({"kind": "smoke", "item": "abnumber/ANARCI", "ok": "X", "detail": f"{type(e).__name__}: {e}"})
    print("X  smoke        실패 —", type(e).__name__, e)
    print("   hmmscan 이 없으면 여기서 FileNotFoundError 가 나요 (본문 3.0)")
    print("   → Colab/Ubuntu: apt-get install -y hmmer   /   로컬: conda install -c bioconda hmmer")

report = pd.DataFrame(rows)
report.to_csv("my_run/setup_report.csv", index=False)
fail = int((report.ok == "X").sum())
print("\nWrote: my_run/setup_report.csv")
print(f"판정 — 점검 {len(report)}건 중 실패 {fail}건.", "이 챕터 스택은 다 섰어요." if fail == 0
      else "실패 항목은 다음 절 표에서 조치 방법과 함께 봅니다.")

## 2) 내 결과 확인 — 점검표와 numbering 속도 (본문 3.0)

본문 3.0 은 hmmscan 만 연결되면 `abnumber.Chain(seq, scheme='imgt')` 가 **0.1초 만에** 돈다고 했죠.
방금 잰 시간으로 그 기준을 직접 확인해요.

In [ ]:
import pathlib
import pandas as pd

try:
    rep, smoke, src = report, SMOKE, "셀 1 (이번 실행 결과를 메모리에서 그대로)"
except NameError:                                   # 커널을 다시 띄운 경우
    p = pathlib.Path("my_run/setup_report.csv")
    rep, smoke, src = (pd.read_csv(p), {}, str(p)) if p.exists() else (None, {}, None)

if rep is None:
    print("점검표가 없어요 — 1절 셀을 먼저 실행하세요.")
    print("(이 파일은 환경마다 값이 달라서 커밋된 대조군이 없어요. 폴백할 data/ 파일도 없습니다.)")
else:
    print("[출처]", src)
    display(rep)
    missing = rep[rep.ok == "X"]
    if missing.empty:
        print(f"판정 — 점검 {len(rep)}건 모두 통과.")
    else:
        print(f"판정 — {len(missing)}건이 미해결이에요. 아래를 먼저 해결하세요.")
        for _, r in missing.iterrows():
            print("  -", r["item"], "|", r["detail"])

if smoke:
    for sid, v in smoke.items():
        print(f"   numbering {sid:8s} {v['sec']:.2f}초 ({v['length']} aa)")
    fast = min(v["sec"] for v in smoke.values())
    print(f"판정 — 가장 빠른 호출 {fast:.2f}초 / 본문 3.0 기준 0.1초 →",
          "기준 안이에요." if fast <= 0.1
          else "기준보다 느려요. 첫 호출에 HMM 프로파일 로딩이 같이 들어간 값이면 정상이고, 두 사슬 다 느리면 hmmscan 경로를 확인하세요.")

## 3) 레퍼런스 대조 — numbering 결과가 정답과 같은가 (본문 3.5)

`data/setup_expected.csv` 에는 같은 데모 항체를 ANARCI(IMGT)로 돌렸을 때 **나와야 하는 값**이 들어 있어요.
스모크 테스트 결과가 이것과 같으면 환경이 제대로 선 거예요.

In [ ]:
import pathlib
import pandas as pd

exp_p = pathlib.Path("data/setup_expected.csv")
try:
    smoke
except NameError:
    smoke = {}

if not exp_p.exists():
    print("정답지가 없어 대조를 건너뜁니다:", exp_p)
elif not smoke:
    print("스모크 테스트 결과가 없어 대조를 건너뜁니다 — 1절 셀을 먼저 실행하세요.")
else:
    exp = pd.read_csv(exp_p)
    display(exp)
    keys = [k for k in ["chain_type", "length", "cdr1_imgt", "cdr2_imgt", "cdr3_imgt"] if k in exp.columns]
    hit = 0
    for _, r in exp.iterrows():
        mine = smoke.get(r["id"])
        if mine is None:
            print("불일치", r["id"], "— 내 결과에 이 서열이 없어요"); continue
        bad = [k for k in keys if str(mine.get(k)) != str(r[k])]
        hit += (not bad)
        detail = "" if not bad else "  ← " + ", ".join(f"{k}: 기대 {r[k]} / 내 결과 {mine.get(k)}" for k in bad)
        print(("일치  " if not bad else "불일치"), r["id"], "|",
              " ".join(f"{k}={r[k]}" for k in keys), detail)
    print(f"\n판정 — 정답지 {len(exp)}개 중 {hit}개 일치({len(keys)}개 항목 전수 비교).",
          "환경이 제대로 섰어요." if hit == len(exp)
          else "한 건이라도 다르면 hmmscan·ANARCI 버전을 먼저 확인하세요 — 보고서엔 도구 버전을 함께 적습니다.")

## 4) 로컬 재현용 conda 환경 파일 점검 (본문 3.1~3.4)

Colab 은 노트북마다 런타임이 달라 의존성 충돌이 자연히 없지만, 로컬에서 반복 실행하려면
환경을 셋으로 나눠야 해요(`abseq`·`abstruct`·`abinterface`). 그 정의 파일이 저장소에 실제로
들어 있는지, **버전 고정이 살아 있는지** 확인합니다.

In [ ]:
import pandas as pd

EXPECT = {                                   # 파일 → 반드시 들어 있어야 하는 항목 (본문 3.2~3.4)
    "abseq.yml":       ["hmmer", "anarci", "abnumber", "sapiens"],
    "abstruct.yml":    ["igfold", "transformers==4.36.2", "anarci", "abnumber"],
    "abinterface.yml": ["freesasa", "mdanalysis", "plip"],
}
env_dir = ADV_ROOT / "environment"
rows = []
for fn, keys in EXPECT.items():
    p = env_dir / fn
    txt = p.read_text(encoding="utf-8") if p.exists() else ""
    rows.append({"파일": fn, "존재": "O" if p.exists() else "X",
                 "확인 항목": len(keys), "빠진 항목": ", ".join(k for k in keys if k not in txt) or "없음"})
envs = pd.DataFrame(rows)
display(envs)

ok = (envs["존재"] == "O").all() and (envs["빠진 항목"] == "없음").all()
print(f"판정 — 환경 파일 {len(envs)}개 중 존재 {int((envs['존재'] == 'O').sum())}개 ·",
      "필요한 항목이 전부 들어 있어요." if ok else "위 표의 '빠진 항목' 을 채워야 해요.")
print("      transformers==4.36.2 고정은 IgFold 체크포인트에 pickle 된 옛 토크나이저 때문이에요(본문 3.3).")
print("      PyMOL 은 pip·conda 어느 쪽으로도 이 목록에 못 들어가요 — Ch.06·07 의 3D 렌더만 커밋 이미지로 대체합니다(본문 3.0).")

> 다음 → 본문 [04. numbering & germline](../04_numbering/04_numbering.md)